In [11]:
from util import State, Step, SearchProblem, UniformCostSearch
from map_util import CityMap

class ShortestPathProblem(SearchProblem):
    def __init__(self, city_map, start_location, end_tag):
        self.city_map = city_map
        self.start_location = start_location
        self.end_tag = end_tag

    def start_state(self):
        return State(location=self.start_location)

    def successors(self, state):
        result = []
        for neighbor, distance in self.city_map.distances[state.location].items():
            result.append(
                Step(
                    action=neighbor,
                    cost=distance,
                    state=State(location=neighbor)
                )
            )
        return result

    def is_end(self, state):
        return self.end_tag in self.city_map.tags[state.location]

In [12]:
from map_util import create_grid_map_with_custom_tags

tags = {(x, y): [] for x in range(5) for y in range(5)}
tags[(0, 0)] = ["landmark=start"]
tags[(4, 1)] = ["amenity=food"]
tags[(2, 4)] = ["amenity=food"]
tags[(3, 3)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(5, 5, tags)
city_map

In [13]:
print(city_map.geo_locations.keys())
print(city_map.tags)
print(city_map.distances["0,0"])

dict_keys(['0,0', '0,1', '0,2', '0,3', '0,4', '1,0', '1,1', '1,2', '1,3', '1,4', '2,0', '2,1', '2,2', '2,3', '2,4', '3,0', '3,1', '3,2', '3,3', '3,4', '4,0', '4,1', '4,2', '4,3', '4,4'])
defaultdict(<class 'list'>, {'0,0': ['label=0,0', 'x=0', 'y=0', 'landmark=start'], '0,1': ['label=0,1', 'x=0', 'y=1'], '0,2': ['label=0,2', 'x=0', 'y=2'], '0,3': ['label=0,3', 'x=0', 'y=3'], '0,4': ['label=0,4', 'x=0', 'y=4'], '1,0': ['label=1,0', 'x=1', 'y=0'], '1,1': ['label=1,1', 'x=1', 'y=1'], '1,2': ['label=1,2', 'x=1', 'y=2'], '1,3': ['label=1,3', 'x=1', 'y=3'], '1,4': ['label=1,4', 'x=1', 'y=4'], '2,0': ['label=2,0', 'x=2', 'y=0'], '2,1': ['label=2,1', 'x=2', 'y=1'], '2,2': ['label=2,2', 'x=2', 'y=2'], '2,3': ['label=2,3', 'x=2', 'y=3'], '2,4': ['label=2,4', 'x=2', 'y=4', 'amenity=food'], '3,0': ['label=3,0', 'x=3', 'y=0'], '3,1': ['label=3,1', 'x=3', 'y=1'], '3,2': ['label=3,2', 'x=3', 'y=2'], '3,3': ['label=3,3', 'x=3', 'y=3', 'parking=garage'], '3,4': ['label=3,4', 'x=3', 'y=4'], '4,0': ['lab

In [14]:
print(city_map.tags["0,0"])
print(city_map.tags["4,1"])
print(city_map.tags["2,4"])

['label=0,0', 'x=0', 'y=0', 'landmark=start']
['label=4,1', 'x=4', 'y=1', 'amenity=food']
['label=2,4', 'x=2', 'y=4', 'amenity=food']


In [15]:
problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

In [16]:
from util import UniformCostSearch

problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

ucs = UniformCostSearch(verbose=1)
ucs.solve(problem)

print("Actions:", ucs.actions)
print("Path cost:", ucs.path_cost)
print("States explored:", ucs.num_states_explored)

num_states_explored = 19
path_cost = 5.0
actions = ['0,1', '1,1', '2,1', '3,1', '4,1']
Actions: ['0,1', '1,1', '2,1', '3,1', '4,1']
Path cost: 5.0
States explored: 19


In [17]:
class WaypointsShortestPathProblem(SearchProblem):
    def __init__(self, city_map, start_location, end_tag, waypoint_tags):
        self.city_map = city_map
        self.start_location = start_location
        self.end_tag = end_tag
        self.waypoint_tags = waypoint_tags

    def start_state(self):
        done = []
        for tag in self.waypoint_tags:
            if tag in self.city_map.tags[self.start_location]:
                done.append(tag)
        return State(location=self.start_location, memory=tuple(sorted(done)))

    def successors(self, state):
        result = []
        for neighbor, distance in self.city_map.distances[state.location].items():
            done = set(state.memory)
            for tag in self.waypoint_tags:
                if tag in self.city_map.tags[neighbor]:
                    done.add(tag)
            new_memory = tuple(sorted(done))
            result.append(
                Step(
                    action=neighbor,
                    cost=distance,
                    state=State(location=neighbor, memory=new_memory)
                    )
                    )
        return result

    def is_end(self, state):
        return (
            self.end_tag in self.city_map.tags[state.location]
            and set(self.waypoint_tags).issubset(set(state.memory))
            )

In [18]:
from map_util import create_grid_map_with_custom_tags, check_valid, get_total_cost
from util import UniformCostSearch, State

# 6x6 synthetic grid
tags = {(x, y): [] for x in range(6) for y in range(6)}

# Start and goal
tags[(0, 0)] = ["landmark=start"]
tags[(5, 2)] = ["landmark=goal"]

# Waypoints
# This one location satisfies BOTH waypoint tags at once
tags[(2, 2)] = ["amenity=food", "parking=garage"]

# Extra distractors so the map is not trivial
tags[(1, 4)] = ["amenity=food"]
tags[(4, 0)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(6, 6, tags)

start_location = "0,0"
end_tag = "landmark=goal"
waypoint_tags = ["amenity=food", "parking=garage"]

problem = WaypointsShortestPathProblem(
    city_map=city_map,
    start_location=start_location,
    end_tag=end_tag,
    waypoint_tags=waypoint_tags,
)

ucs = UniformCostSearch(verbose=1)
ucs.solve(problem)

path = [start_location] + ucs.actions

print("Actions:", ucs.actions)
print("Path cost:", ucs.path_cost)
print("States explored:", ucs.num_states_explored)
print("Path:", path)
print("Valid path?", check_valid(path, city_map, start_location, end_tag, waypoint_tags))
print("Recomputed total cost:", get_total_cost(path, city_map))

num_states_explored = 73
path_cost = 7.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Path cost: 7.0
States explored: 73
Path: ['0,0', '0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Valid path? True
Recomputed total cost: 7.0


In [19]:
def a_star_reduction(problem, heuristic):
    class ReducedProblem(SearchProblem):
        def start_state(self):
            return problem.start_state()

        def is_end(self, state):
            return problem.is_end(state)

        def successors(self, state):
            result = []
            for step in problem.successors(state):
                new_cost = (
                    step.cost
                    + heuristic.evaluate(step.state)
                    - heuristic.evaluate(state)
                )
                result.append(
                    Step(
                        action=step.action,
                        cost=new_cost,
                        state=step.state
                    )
                )
            return result

    return ReducedProblem()

In [20]:
from util import Heuristic, UniformCostSearch

class ZeroHeuristic(Heuristic):
    def evaluate(self, state):
        return 0.0

zero_heuristic = ZeroHeuristic()

# Use one of your existing synthetic problems
problem = ShortestPathProblem(city_map, "0,0", "amenity=food")

# Ordinary UCS
ucs_original = UniformCostSearch(verbose=1)
ucs_original.solve(problem)

print("Original UCS actions:", ucs_original.actions)
print("Original UCS path cost:", ucs_original.path_cost)
print("Original UCS states explored:", ucs_original.num_states_explored)

# UCS on the A*-reduced problem
reduced_problem = a_star_reduction(problem, zero_heuristic)

ucs_reduced = UniformCostSearch(verbose=1)
ucs_reduced.solve(reduced_problem)

print("Reduced UCS actions:", ucs_reduced.actions)
print("Reduced UCS path cost:", ucs_reduced.path_cost)
print("Reduced UCS states explored:", ucs_reduced.num_states_explored)

num_states_explored = 13
path_cost = 4.0
actions = ['0,1', '0,2', '1,2', '2,2']
Original UCS actions: ['0,1', '0,2', '1,2', '2,2']
Original UCS path cost: 4.0
Original UCS states explored: 13
num_states_explored = 13
path_cost = 4.0
actions = ['0,1', '0,2', '1,2', '2,2']
Reduced UCS actions: ['0,1', '0,2', '1,2', '2,2']
Reduced UCS path cost: 4.0
Reduced UCS states explored: 13


In [21]:
from util import Heuristic
from map_util import compute_distance

class StraightLineHeuristic(Heuristic):
    def __init__(self, city_map, end_tag):
        self.city_map = city_map
        self.end_tag = end_tag

        self.end_locations = []
        for location, tags in self.city_map.tags.items():
            if self.end_tag in tags:
                self.end_locations.append(location)

    def evaluate(self, state):
        current_geo = self.city_map.geo_locations[state.location]

        distances = []
        for end_location in self.end_locations:
            end_geo = self.city_map.geo_locations[end_location]
            distances.append(compute_distance(current_geo, end_geo))

        return min(distances)
        

In [26]:
from map_util import create_grid_map_with_custom_tags

tags = {(x, y): [] for x in range(6) for y in range(6)}
tags[(0, 0)] = ["landmark=start"]
tags[(5, 2)] = ["landmark=goal"]
tags[(2, 2)] = ["amenity=food", "parking=garage"]
tags[(1, 4)] = ["amenity=food"]
tags[(4, 0)] = ["parking=garage"]

city_map = create_grid_map_with_custom_tags(6, 6, tags)

heuristic = StraightLineHeuristic(city_map, "landmark=goal")
print(heuristic.evaluate(State(location="0,0")))
print(heuristic.evaluate(State(location="5,2")))

5.9880300569817
0.0


In [28]:
straight_line_heuristic = StraightLineHeuristic(city_map, "amenity=food")
reduced_problem = a_star_reduction(problem, straight_line_heuristic)

ucs_astar = UniformCostSearch(verbose=1)
ucs_astar.solve(reduced_problem)

print("A* via reduction actions:", ucs_astar.actions)
print("A* via reduction reduced-cost:", ucs_astar.path_cost)
print("A* via reduction states explored:", ucs_astar.num_states_explored)

from map_util import get_total_cost
path = [problem.start_state().location] + ucs_astar.actions
print("Returned path:", path)
print("Original path cost:", get_total_cost(path, city_map))

num_states_explored = 6
path_cost = 0.8549325334437041
actions = ['1,0', '1,1', '2,1', '2,2']
A* via reduction actions: ['1,0', '1,1', '2,1', '2,2']
A* via reduction reduced-cost: 0.8549325334437041
A* via reduction states explored: 6
Returned path: ['0,0', '1,0', '1,1', '2,1', '2,2']
Original path cost: 4.0


In [29]:
from util import Heuristic, UniformCostSearch

class NoWaypointsHeuristic(Heuristic):
    def __init__(self, city_map, end_tag):
        self.city_map = city_map
        self.end_tag = end_tag
        self.cost_to_goal = {}

        for location in self.city_map.geo_locations.keys():
            problem = ShortestPathProblem(
                city_map=self.city_map,
                start_location=location,
                end_tag=self.end_tag
            )
            ucs = UniformCostSearch(verbose=0)
            ucs.solve(problem)
            self.cost_to_goal[location] = ucs.path_cost

    def evaluate(self, state):
        return self.cost_to_goal[state.location]

In [30]:
no_wp_heuristic = NoWaypointsHeuristic(city_map, "landmark=goal")

print(no_wp_heuristic.evaluate(State(location="0,0")))
print(no_wp_heuristic.evaluate(State(location="5,2")))
print(no_wp_heuristic.evaluate(State(location="2,2")))

7.0
0.0
3.0


In [31]:
waypoint_problem = WaypointsShortestPathProblem(
    city_map=city_map,
    start_location="0,0",
    end_tag="landmark=goal",
    waypoint_tags=["amenity=food", "parking=garage"]
)

# 1) Plain UCS
ucs_plain = UniformCostSearch(verbose=1)
ucs_plain.solve(waypoint_problem)

print("Plain UCS actions:", ucs_plain.actions)
print("Plain UCS reduced/original cost:", ucs_plain.path_cost)
print("Plain UCS explored:", ucs_plain.num_states_explored)

# 2) A* with straight-line heuristic
sl_heuristic = StraightLineHeuristic(city_map, "landmark=goal")
sl_reduced = a_star_reduction(waypoint_problem, sl_heuristic)

ucs_sl = UniformCostSearch(verbose=1)
ucs_sl.solve(sl_reduced)

print("A* straight-line actions:", ucs_sl.actions)
print("A* straight-line reduced cost:", ucs_sl.path_cost)
print("A* straight-line explored:", ucs_sl.num_states_explored)

# 3) A* with no-waypoints heuristic
nw_heuristic = NoWaypointsHeuristic(city_map, "landmark=goal")
nw_reduced = a_star_reduction(waypoint_problem, nw_heuristic)

ucs_nw = UniformCostSearch(verbose=1)
ucs_nw.solve(nw_reduced)

print("A* no-waypoints actions:", ucs_nw.actions)
print("A* no-waypoints reduced cost:", ucs_nw.path_cost)
print("A* no-waypoints explored:", ucs_nw.num_states_explored)

# Recompute original path costs for the two A* runs
from map_util import get_total_cost

path_sl = ["0,0"] + ucs_sl.actions
path_nw = ["0,0"] + ucs_nw.actions

print("A* straight-line original cost:", get_total_cost(path_sl, city_map))
print("A* no-waypoints original cost:", get_total_cost(path_nw, city_map))

num_states_explored = 73
path_cost = 7.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Plain UCS actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
Plain UCS reduced/original cost: 7.0
Plain UCS explored: 73
num_states_explored = 23
path_cost = 1.0119699430182998
actions = ['1,0', '2,0', '2,1', '2,2', '3,2', '4,2', '5,2']
A* straight-line actions: ['1,0', '2,0', '2,1', '2,2', '3,2', '4,2', '5,2']
A* straight-line reduced cost: 1.0119699430182998
A* straight-line explored: 23
num_states_explored = 24
path_cost = 0.0
actions = ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
A* no-waypoints actions: ['0,1', '0,2', '1,2', '2,2', '3,2', '4,2', '5,2']
A* no-waypoints reduced cost: 0.0
A* no-waypoints explored: 24
A* straight-line original cost: 7.0
A* no-waypoints original cost: 7.0
